In [34]:
from dotenv import load_dotenv
import os
load_dotenv()


True

In [35]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"  # adjust if needed

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

Loaded 9 pages from the PDF.

--- First page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,     
    chunk_overlap=100,   
    separators=["\n\n", "\n", ".", " "], 
)

chunks = splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}  (from {len(pages)} pages)")
print(f"Avg chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print("\n--- Example chunk ---")
print(chunks[5].page_content)

Total chunks: 37  (from 9 pages)
Avg chunk length: 504 chars

--- Example chunk ---
Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approach
resolves the majority of cases without escalation.
Step 1  - Verify signal strength. Open the device's status bar or dial *3001#12345#* (iOS) or use a network signal
app (Android) to view the raw signal level in dBm. A signal above -85 dBm is good; between -85 and -100 dBm is
marginal; below -100 dBm is poor. If signal is weak, moving closer to a window or to a higher floor often helps.


In [37]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Loading embedding model (downloads ~90 MB on first run)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embedding all chunks and storing in Chroma (in memory)...")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")


Loading embedding model (downloads ~90 MB on first run)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2063.78it/s]


Embedding all chunks and storing in Chroma (in memory)...
Vector store ready. 185 vectors stored.


In [38]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved)} chunks:\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()


Query: What is VoLTE and how does it improve call quality?
Retrieved 3 chunks:

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 3 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.

In [39]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI


# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


# --- System prompt: ground the LLM in the retrieved context ---
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)


# --- Assemble the chain ---
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled.")

RAG chain assembled.
